In [ ]:
import xpress as xp
import pandas as pd
import numpy as np
from datetime import datetime

class TimetablingModel:
    """Full University Timetabling Model: Redistribution + Strict Weeks + Stability."""
    
    def __init__(self, students_df, events_df, weeks_df, rooms_df):
        self.students_df = students_df
        self.events_df = events_df
        self.weeks_df = weeks_df
        self.rooms_df = rooms_df
        
        # Initialize Xpress
        xp.init()
        self.model = xp.problem(name="Timetabling")
        
        # Dimensions
        self.events, self.weeks, self.rooms = [], [], []
        self.days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
        self.time_slots = [f"{h:02d}:{m:02d}" for h in range(9, 18) for m in [0, 30]]
        
        # Data Mappings
        self.event_size, self.event_duration, self.event_weeks = {}, {}, {}
        self.event_name, self.room_capacity, self.room_campus = {}, {}, {}
        self.room_building, self.curricula = {}, {}
        self.event_original_room, self.event_original_campus = {}, {}
        
        self.x, self.v = {}, {}
        print("✓ Model initialized")

    def build_data(self, max_events=70, max_rooms=30):
        print("Building data structures...")
        
        # 1. Weeks (Semester Calendar)
        teaching_weeks = self.weeks_df[self.weeks_df['Week Type'] == 'Other']['Week Number'].unique()
        if len(teaching_weeks) == 0: teaching_weeks = self.weeks_df['Week Number'].unique()
        self.weeks = sorted([int(w) for w in teaching_weeks if pd.notna(w)])

        # Helper for Week Parsing (handles '10-12' or '9,11')
        def parse_weeks(week_str):
            if pd.isna(week_str): return self.weeks
            w_set = set()
            for part in str(week_str).split(','):
                part = part.strip()
                if '-' in part:
                    s, e = part.split('-')
                    w_set.update(range(int(s), int(e) + 1))
                else:
                    w_set.add(int(part))
            return sorted([w for w in w_set if w in self.weeks])

        # 2. Events (The Classes)
        self.events = self.events_df['Event ID'].unique()[:max_events].tolist()
        for e in self.events:
            row = self.events_df[self.events_df['Event ID'] == e].iloc[0]
            self.event_name[e] = row.get('Event Name', 'Unknown')
            self.event_size[e] = int(row.get('Event Size', 0)) if pd.notna(row.get('Event Size')) else 0
            
            # Duration: blocks of 30 mins
            dur = row.get('Duration (minutes)', 60)
            self.event_duration[e] = max(1, int(np.ceil(float(dur) / 30)))
            
            # CRITICAL: Specific weeks for this event
            self.event_weeks[e] = parse_weeks(row.get('Weeks'))
            
            # Stability Info
            self.event_original_room[e] = row.get('Room')
            self.event_original_campus[e] = row.get('Campus', 'Unknown')

        # 3. Rooms (Excluding Holyrood for Redistribution)
        room_count = 0
        for _, row in self.rooms_df.iterrows():
            if room_count >= max_rooms: break
            if row.get('Campus') == 'Holyrood': continue
            r_id = row['Id']
            self.rooms.append(r_id)
            self.room_capacity[r_id] = int(row.get('Capacity', 0))
            self.room_campus[r_id] = row.get('Campus', 'Central')
            self.room_building[r_id] = row.get('Building_Name', 'Unknown')
            room_count += 1

        # 4. Curricula (Clash Prevention)
        student_events = self.students_df.groupby('AnonID')['Event ID'].apply(set).to_dict()
        c_id = 0
        for events in list(student_events.values())[:5000]:
            relevant = [e for e in events if e in self.events]
            if len(relevant) > 1:
                self.curricula[c_id] = relevant
                c_id += 1
        print(f"✓ Data built: {len(self.events)} events, {len(self.rooms)} rooms")


        # 5. Load Travel Matrix from your CSV headers
        print("Loading travel time matrix...")
        try:
            # Reading the CSV with headers from your image
            travel_df = pd.read_csv('Travel_Times.csv') 
            self.travel_matrix = {}
            for _, row in travel_df.iterrows():
                c_from = row['Campus From']
                c_to = row['Campus To']
                mins = row['Travel time (mins)']
                
                # Convert minutes to number of 30-min slots needed
                # 60 mins = 2 slots gap
                slots = int(np.ceil(mins / 30))
                
                self.travel_matrix[(c_from, c_to)] = slots
                # Ensure the model knows the time is the same both ways
                self.travel_matrix[(c_to, c_from)] = slots
        except Exception as e:
            print(f"Warning: Could not load Travel_Times.csv: {e}")
            self.travel_matrix = {}

        print(f"✓ Travel data loaded for {len(self.travel_matrix)} campus pairs.")


    def build_model(self):
        print("Building MIP model (9AM-6PM Strict, Stability Prioritised)...")
        
        # 1. Variables (Only created for event-specific weeks)
        for e in self.events:
            for w in self.event_weeks.get(e, []):
                for d in self.days:
                    for t in self.time_slots:
                        for r in self.rooms:
                            self.x[e,d,t,r,w] = xp.var(vartype=xp.binary, name=f"x_{e}_{d}_{t}_{r}_{w}")
                            self.v[e,d,t,r,w] = xp.var(lb=0)
        
        self.model.addVariable(list(self.x.values()) + list(self.v.values()))

        # 2. Penalty Calculations (Objective)
        campus_penalties = []
        relocation_penalties = []
        room_stability_penalties = []
        
        for (e, d, t, r, w), x_var in self.x.items():
            camp = self.room_campus.get(r, 'Central')
            orig_room = self.event_original_room.get(e)
            orig_camp = self.event_original_campus.get(e)
            
            # Campus Weightings
            p_r = 10 if camp == 'Lauriston' else (20 if camp == 'New College' else 0)
            if p_r > 0: campus_penalties.append(p_r * x_var)
            
            # Redistribution Stability (Ignore Holyrood as it must move)
            if orig_camp != 'Holyrood':
                if r != orig_room:
                    room_stability_penalties.append(25 * x_var)
                if camp != orig_camp:
                    relocation_penalties.append(50 * x_var)

        cap_obj = xp.Sum(100 * self.v[e,d,t,r,w] for (e,d,t,r,w) in self.x)
        
        self.model.setObjective(
            cap_obj + 
            xp.Sum(campus_penalties) + 
            xp.Sum(relocation_penalties) + 
            xp.Sum(room_stability_penalties),
            sense=xp.minimize
        )

        # 3. Constraints
        # A. Completeness (Strictly on assigned weeks)
        for e in self.events:
            for w in self.event_weeks.get(e, []):
                self.model.addConstraint(xp.Sum(self.x[e,d,t,r,w] 
                    for d in self.days for t in self.time_slots for r in self.rooms 
                    if (e,d,t,r,w) in self.x) == 1)

        # B. 6 PM Boundary (Duration awareness)
        for e in self.events:
            L_e = self.event_duration[e]
            for t_idx, t in enumerate(self.time_slots):
                if t_idx + L_e > len(self.time_slots):
                    target_vars = [self.x[e,d,t,r,w] for d in self.days for r in self.rooms 
                                   for w in self.event_weeks.get(e, []) if (e,d,t,r,w) in self.x]
                    if target_vars:
                        self.model.addConstraint(xp.Sum(target_vars) == 0)

        # C. Room Uniqueness
        for r in self.rooms:
            for w in self.weeks:
                for d in self.days:
                    for t_idx, t in enumerate(self.time_slots):
                        occ = []
                        for e in self.events:
                            if w in self.event_weeks.get(e, []):
                                dur = self.event_duration[e]
                                for k in range(dur):
                                    if t_idx - k >= 0:
                                        t_p = self.time_slots[t_idx - k]
                                        if (e,d,t_p,r,w) in self.x: occ.append(self.x[e,d,t_p,r,w])
                        if occ: self.model.addConstraint(xp.Sum(occ) <= 1)

        # D. Curriculum Clash
        for events in self.curricula.values():
            for w in self.weeks:
                for d in self.days:
                    for t_idx, t in enumerate(self.time_slots):
                        clash = []
                        for e in events:
                            if w in self.event_weeks.get(e, []):
                                dur = self.event_duration[e]
                                for k in range(dur):
                                    if t_idx - k >= 0:
                                        t_p = self.time_slots[t_idx - k]
                                        for r in self.rooms:
                                            if (e,d,t_p,r,w) in self.x: clash.append(self.x[e,d,t_p,r,w])
                        if clash: self.model.addConstraint(xp.Sum(clash) <= 1)

        # E. Capacity Logic
        for (e,d,t,r,w), x_v in self.x.items():
            sz, cp, v_v = self.event_size[e], self.room_capacity[r], self.v[e,d,t,r,w]
            self.model.addConstraint(v_v >= (0.5 * cp - sz) * x_v)
            self.model.addConstraint(v_v >= (sz - cp) * x_v)
        
        print(f"✓ Model built with {len(self.x)} binary variables.")

        # D. Duration-Aware Curriculum Clash + Travel Time
        print("Applying Travel & Clash Constraints...")
        for events in self.curricula.values():
            for w in self.weeks:
                for d in self.days:
                    for t_idx, t in enumerate(self.time_slots):
                        
                        # 1. Standard Clash (Cannot be in two classes at the exact same time)
                        clash_vars = []
                        for e in events:
                            if w in self.event_weeks.get(e, []):
                                for k in range(self.event_duration[e]):
                                    if t_idx - k >= 0:
                                        t_p = self.time_slots[t_idx - k]
                                        for r in self.rooms:
                                            if (e,d,t_p,r,w) in self.x:
                                                clash_vars.append(self.x[e,d,t_p,r,w])
                        if clash_vars:
                            self.model.addConstraint(xp.Sum(clash_vars) <= 1)

                        # 2. Travel Gap (Cannot start next class too soon after previous one ends)
                        for e1 in events:
                            if w not in self.event_weeks.get(e1, []): continue
                            L_e1 = self.event_duration[e1]
                            
                            for r1 in self.rooms:
                                camp1 = self.room_campus.get(r1)
                                
                                for e2 in events:
                                    if e1 == e2 or w not in self.event_weeks.get(e2, []): continue
                                    
                                    for r2 in self.rooms:
                                        camp2 = self.room_campus.get(r2)
                                        
                                        # Get required gap slots from your CSV data
                                        req_gap = self.travel_matrix.get((camp1, camp2), 0)
                                        
                                        if req_gap > 0:
                                            # If e1 is at r1 starting at t_idx, 
                                            # e2 cannot start at r2 during the 'travel window'
                                            for gap_fill in range(req_gap):
                                                t_illegal_idx = t_idx + L_e1 + gap_fill
                                                if t_illegal_idx < len(self.time_slots):
                                                    t_illegal = self.time_slots[t_illegal_idx]
                                                    
                                                    # Binary constraint: If e1 is here, e2 cannot be there
                                                    if (e1,d,t,r1,w) in self.x and (e2,d,t_illegal,r2,w) in self.x:
                                                        self.model.addConstraint(self.x[e1,d,t,r1,w] + self.x[e2,d,t_illegal,r2,w] <= 1)


    def solve(self, time_limit=300):
        print(f"Solving (Limit: {time_limit}s)...")
        self.model.controls.maxtime = -time_limit
        self.model.solve()
        return self.model.attributes.solstatus in [xp.SolStatus.FEASIBLE, xp.SolStatus.OPTIMAL]

    
    def extract_solution(self):
        print("🚀 Extracting detailed schedule with capacity data...")
        vars_list = list(self.x.values())
        keys_list = list(self.x.keys())
        sol_values = self.model.getSolution(vars_list)
        
        results = []
        for i, (e_id, d, t, r_id, w) in enumerate(keys_list):
            if sol_values[i] > 0.5:
                results.append({
                    'Week': w, 
                    'Day': d, 
                    'Time': t, 
                    'Event_ID': e_id,
                    'Event_Name': self.event_name.get(e_id),
                    'Event_Size': self.event_size.get(e_id),      # Added back
                    'Sched_Room': r_id, 
                    'Room_Capacity': self.room_capacity.get(r_id), # Added back
                    'Sched_Building': self.room_building.get(r_id),
                    'Sched_Campus': self.room_campus.get(r_id),
                    'Orig_Room': self.event_original_room.get(e_id),
                    'Orig_Campus': self.event_original_campus.get(e_id),
                    'Was_Relocated': 'Yes' if r_id != self.event_original_room.get(e_id) else 'No'
                })
        
        df = pd.DataFrame(results)
        if not df.empty:
            day_map = {d: i for i, d in enumerate(self.days)}
            df['d_idx'] = df['Day'].map(day_map)
            # Sort chronologically
            df = df.sort_values(['Week', 'd_idx', 'Time']).drop('d_idx', axis=1)
            
            # Print a quick summary of the redistribution success
            print(f"✓ Extracted {len(df)} events.")
            relocated_count = df[df['Was_Relocated'] == 'Yes'].shape[0]
            print(f"✓ Total Relocations: {relocated_count}")
            
        return df

def run_optimization():
    try:
        e_df, r_df = pd.read_csv('Event_data.csv'), pd.read_csv('Rooms_data.csv')
        s_df, st_df = pd.read_csv('Semester1.csv'), pd.read_csv('Student_data.csv')
    except Exception as err:
        print(f"File Error: {err}"); return

    model = TimetablingModel(st_df, e_df, s_df, r_df)
    # Adjust max_events and max_rooms here or use len(e_df) / len(r_df)
    model.build_data(max_events=100, max_rooms=20)
    model.build_model()
    
    if model.solve():
        df = model.extract_solution()
        df.to_csv('final_redistribution_schedule1.csv', index=False)
        print(f"✅ Success! Saved to final_redistribution_schedule.csv")
    else:
        print("❌ No feasible solution found.")

if __name__ == "__main__":
    run_optimization()

✓ Model initialized
Building data structures...
✓ Data built: 100 events, 20 rooms
Loading travel time matrix...
✓ Travel data loaded for 64 campus pairs.
Building MIP model (9AM-6PM Strict, Stability Prioritised)...


C:\Users\Selina Kanguha\AppData\Local\Temp\ipykernel_2316\2926191842.py:128: DeprecationWarning: Deprecated in Xpress 9.5: create a linked variable by calling problem.addVariable()
  self.x[e,d,t,r,w] = xp.var(vartype=xp.binary, name=f"x_{e}_{d}_{t}_{r}_{w}")
C:\Users\Selina Kanguha\AppData\Local\Temp\ipykernel_2316\2926191842.py:129: DeprecationWarning: Deprecated in Xpress 9.5: create a linked variable by calling problem.addVariable()
  self.v[e,d,t,r,w] = xp.var(lb=0)


✓ Model built with 1249200 binary variables.
Applying Travel & Clash Constraints...
Solving (Limit: 300s)...
FICO Xpress v9.7.0, Hyper, solve started 0:44:57, Mar 22, 2026
Heap usage: 1583MB (peak 1583MB, 66MB system)
Minimizing MILP Timetabling using up to 22 threads and up to 15GB memory, with these control settings:
MAXTIME = -300
OUTPUTLOG = 1
NLPPOSTSOLVE = 1
XSLP_DELETIONCONTROL = 0
XSLP_OBJSENSE = 1
Original problem has:
   2560640 rows      2498400 cols     13323270 elements   1249200 entities
Presolved problem has:
     22644 rows      1138900 cols      4427100 elements   1138900 entities
Presolve finished in 70 seconds
Heap usage: 2411MB (peak 4122MB, 66MB system)

Coefficient range                    original                 solved        
  Coefficients   [min,max] : [ 5.00e-01,  3.94e+02] / [ 1.00e+00,  1.00e+00]
  RHS and bounds [min,max] : [ 1.00e+00,  1.00e+00] / [ 1.00e+00,  1.00e+00]
  Objective      [min,max] : [ 2.50e+01,  1.00e+02] / [ 2.50e+01,  3.88e+04]
Autoscal